In [1]:
import sys
sys.path.append("../..")  # raíz del repo, para poder importar src.*

import json
import os

import numpy as np
import pandas as pd
import tensorflow as tf

from src.models.gru.model import build_gru_model
from src.models.gru.train import compute_class_weights, train_gru_model
from src.models.gru.threshold import sweep_thresholds, select_threshold_strategies

np.random.seed(42)
tf.random.set_seed(42)

In [2]:
DATA_DIR = "../../data/sequences"
W = 20

X_train = np.load(f"{DATA_DIR}/X_train_w{W}.npy")
y_train = np.load(f"{DATA_DIR}/y_train_w{W}.npy")
X_val = np.load(f"{DATA_DIR}/X_val_w{W}.npy")
y_val = np.load(f"{DATA_DIR}/y_val_w{W}.npy")

print(f"X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"X_val:   {X_val.shape} | y_val:   {y_val.shape}")

X_train: (315172, 20, 63) | y_train: (315172,)
X_val:   (67463, 20, 63) | y_val:   (67463,)


In [3]:
class_weight_dict = compute_class_weights(y_train)
print("class_weight:", class_weight_dict)

class_weight: {0: 0.5131606798029236, 1: 19.495979215637757}


In [4]:
# Semilla justo antes de construir/entrenar (mismo patrón que train_gru_model),
# para que W=20 sea comparable con el baseline W=10 bajo la misma config de arquitectura.
np.random.seed(42)
tf.random.set_seed(42)

model_w20 = build_gru_model(input_shape=(W, X_train.shape[-1]))
history_w20 = train_gru_model(
    model_w20, X_train, y_train, X_val, y_val, class_weight_dict,
    epochs=50, batch_size=64, random_state=42,
)

MODEL_PATH_W20 = "../../models/gru/w20_w20_baseline_config.keras"
os.makedirs(os.path.dirname(MODEL_PATH_W20), exist_ok=True)
model_w20.save(MODEL_PATH_W20)

Epoch 1/50
4925/4925 ━━━━━━━━━━━━━━━━━━━━ 13s 2ms/step - accuracy: 0.9945 - auc: 0.9994 - loss: 0.0220 - precision: 0.8287 - recall: 0.9925 - val_accuracy: 0.9992 - val_auc: 1.0000 - val_loss: 0.0030 - val_precision: 0.9759 - val_recall: 0.9986
Epoch 2/50
4925/4925 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9992 - auc: 0.9996 - loss: 0.0059 - precision: 0.9703 - recall: 0.9980 - val_accuracy: 0.9994 - val_auc: 1.0000 - val_loss: 0.0018 - val_precision: 0.9905 - val_recall: 0.9891
Epoch 3/50
4925/4925 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9995 - auc: 0.9999 - loss: 0.0035 - precision: 0.9812 - recall: 0.9989 - val_accuracy: 0.9993 - val_auc: 1.0000 - val_loss: 0.0020 - val_precision: 0.9891 - val_recall: 0.9896
Epoch 4/50
4925/4925 ━━━━━━━━━━━━━━━━━━━━ 12s 2ms/step - accuracy: 0.9996 - auc: 0.9999 - loss: 0.0022 - precision: 0.9852 - recall: 0.9993 - val_accuracy: 0.9991 - val_auc: 0.9997 - val_loss: 0.0020 - val_precision: 0.9909 - val_recall: 0.9815
Epoch 5/50
4925/4925

In [5]:
val_loss_w20 = model_w20.evaluate(X_val, y_val, verbose=0)[0]

df_umbrales_w20 = sweep_thresholds(model_w20, X_val, y_val)
estrategias_w20 = select_threshold_strategies(df_umbrales_w20)

resultado_w20 = {
    "configuracion": "baseline_w20",
    "val_loss": val_loss_w20,
    "f2_maximo": estrategias_w20["f2_maximo"]["f2"],
    "umbral_f2_maximo": estrategias_w20["f2_maximo"]["threshold"],
    "f1_maximo": estrategias_w20["f1_maximo"]["f1"],
    "umbral_f1_maximo": estrategias_w20["f1_maximo"]["threshold"],
    "epocas_entrenadas": len(history_w20.history["loss"]),
    "parametros_totales": model_w20.count_params(),
}
print(resultado_w20)

{'configuracion': 'baseline_w20', 'val_loss': 0.0017979202093556523, 'f2_maximo': 0.9944149943203333, 'umbral_f2_maximo': 0.373, 'f1_maximo': 0.9917394382818031, 'umbral_f1_maximo': 0.373, 'epocas_entrenadas': 7, 'parametros_totales': 9345}


In [6]:
# Referencia W=10: entrada "baseline_32" del barrido de hiperparámetros
# (docs/gru/adr/0007-exploracion-hiperparametros-gru.md) — no se reentrena.
with open("../../results/gru/hyperparameter_search_w10.json", encoding="utf-8") as f:
    resultados_w10 = json.load(f)

resultado_w10 = next(r for r in resultados_w10 if r["configuracion"] == "baseline_32")
resultado_w10 = {**resultado_w10, "configuracion": "baseline_w10"}
print(resultado_w10)

{'configuracion': 'baseline_w10', 'val_loss': 0.001350093400105834, 'f2_maximo': 0.9966903073286053, 'umbral_f2_maximo': 0.05, 'f1_maximo': 0.9945536348567369, 'umbral_f1_maximo': 0.337, 'epocas_entrenadas': None, 'parametros_totales': 9345}


In [7]:
tabla_comparativa = pd.DataFrame([resultado_w10, resultado_w20])
print(tabla_comparativa)

  configuracion  val_loss  f2_maximo  umbral_f2_maximo  f1_maximo  \
0  baseline_w10  0.001350   0.996690             0.050   0.994554   
1  baseline_w20  0.001798   0.994415             0.373   0.991739   

   umbral_f1_maximo  epocas_entrenadas  parametros_totales  
0             0.337                NaN                9345  
1             0.373                7.0                9345  


In [8]:
RESULTS_PATH = "../../results/gru/window_comparison.json"
os.makedirs(os.path.dirname(RESULTS_PATH), exist_ok=True)

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump([resultado_w10, resultado_w20], f, indent=2, ensure_ascii=False)

print(f"Resultados guardados en: {RESULTS_PATH}")

Resultados guardados en: ../../results/gru/window_comparison.json


## Regla de decisión

W=20 reemplaza a W=10 como ventana oficial solo si, en validación, mejora el F2 máximo
en al menos 0.001 **y** reduce `val_loss` frente a W=10. Si no cumple ambas condiciones
a la vez, se conserva W=10.